# 07 · AWS SageMaker training

**Auteurs:** _vul namen en concrete bijdragen in vóór indiening_  
**Doel:** Minstens één model reproduceerbaar in AWS trainen en artifacts/metrics terughalen.

Elke codecel wordt voorafgegaan door uitleg. Voer de notebooks in nummervolgorde uit
vanaf de repository-root. Resultaten moeten door het team zelf worden gecontroleerd en
geïnterpreteerd; synthetische CI-data gelden nooit als onderzoeksbewijs.

## Voorwaarden en kosten

Dit notebook draait alleen in een geconfigureerd AWS-account. Controleer regio, IAM-rol,
S3-bucket en budgetalarm. Gebruik eerst `ml.m5.large`, stop endpoints na evaluatie en
commit geen credentials. De lokale Parquet-data wordt naar een eigen versleutelde
bucket geüpload; controleer de Citi Bike data policy.

In [ ]:
# Uitvoeren in SageMaker Studio of na: pip install sagemaker boto3
import sagemaker
from sagemaker.sklearn.estimator import SKLearn
from src.config import PROCESSED_DIR

session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = session.default_bucket()
prefix = "green-wheels/citibike"
train_s3 = session.upload_data(
    str(PROCESSED_DIR / "trips.parquet"), bucket=bucket, key_prefix=f"{prefix}/train"
)
print(train_s3)

## Training job

De AWS-job gebruikt dezelfde code, temporal hold-out en metrics als lokaal; alleen de computeomgeving verandert.

In [ ]:
estimator = SKLearn(
    # Root als source_dir bevat zowel aws/train_entrypoint.py als src/.
    # Zo gebruikt SageMaker exact dezelfde feature- en modelcode als lokaal.
    entry_point="aws/train_entrypoint.py",
    source_dir="..",
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    framework_version="1.2-1",
    py_version="py3",
    hyperparameters={"max-rows": 500000},
    output_path=f"s3://{bucket}/{prefix}/output",
    base_job_name="citibike-duration",
)
estimator.fit({"train": train_s3}, wait=True, logs=True)
print("Model artifact:", estimator.model_data)

## Bewijs en vergelijking

Download `model.tar.gz`, bewaar jobnaam/configuratie en voeg de AWS-metrics als aparte rij toe aan notebook 06. Screenshots alleen als aanvulling, nooit als vervanging van code/config.

In [ ]:
import boto3
description = boto3.client("sagemaker").describe_training_job(
    TrainingJobName=estimator.latest_training_job.name
)
evidence = {
    "job_name": description["TrainingJobName"],
    "status": description["TrainingJobStatus"],
    "instance": description["ResourceConfig"]["InstanceType"],
    "model_artifact": description["ModelArtifacts"]["S3ModelArtifacts"],
    "training_seconds": description.get("TrainingTimeInSeconds"),
}
evidence

## Opruimen

Deze training maakt geen persistent endpoint. Verwijder tijdelijke S3-input en outputs pas nadat benodigde artifacts lokaal zijn veiliggesteld en volgens teambeleid mogen worden verwijderd.